#Prototype Code

In [2]:
# GRADUATE EMPLOYABILITY PREDICTION PROTOTYPE

import numpy as np
import pandas as pd
import joblib
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
from datetime import datetime

# 1. Load the saved model and preprocessing objects

best_lgbm = joblib.load("best_lgbm.pkl")
scaler = joblib.load("scaler.pkl")
label_encoders = joblib.load("label_encoders.pkl")
target_encoder = joblib.load("target_encoder.pkl")
feature_columns = joblib.load("feature_columns.pkl")

try:
    raw_df = df.copy()
except NameError:
    raw_df = None

try:
    processed_df = df_processed.copy()
except NameError:
    processed_df = None


# 2. Helper functions

def dataset_values(column, fallback):
    if raw_df is not None and column in raw_df.columns:
        values = raw_df[column].dropna().astype(str).unique().tolist()
        return sorted(values)
    return fallback

def make_dropdown(options, value=None):
    options = list(options)
    if not options:
        options = ["Unknown"]
    if value is None or value not in options:
        value = options[0]
    return widgets.Dropdown(
        options=options,
        value=value,
        layout=widgets.Layout(width="100%", height="38px"),
        style={"description_width": "0px"}
    )

def get_encoder_classes(column, fallback):
    try:
        return [str(x) for x in label_encoders[column].classes_.tolist()]
    except Exception:
        return fallback

def esc(value):
    return (str(value)
            .replace("&", "&amp;")
            .replace("<", "&lt;")
            .replace(">", "&gt;")
            .replace('"', "&quot;"))


# 3. Dataset-aware input options

country_options = dataset_values(
    "Country_of_Origin",
    ["Australia", "Brazil", "China", "Germany", "India", "Nigeria",
     "Pakistan", "USA", "Vietnam"]
)

education_options = dataset_values(
    "Education_Level",
    ["Bachelor's", "Diploma", "Master's", "PhD"]
)

field_options = dataset_values(
    "Field_of_Study",
    ["Arts", "Business", "Engineering", "Health", "IT", "Social Sciences"]
)

language_options = get_encoder_classes(
    "Language_Proficiency",
    ["Basic", "Intermediate", "Advanced", "Fluent"]
)

visa_options = dataset_values(
    "Visa_Type",
    ["None", "Post-study", "Student", "Work Visa", "Permanent Residency"]
)

gender_options = dataset_values(
    "Gender",
    ["Female", "Male", "Other"]
)

ranking_options = get_encoder_classes(
    "University_Ranking",
    ["High", "Medium", "Low"]
)

region_options = dataset_values(
    "Region_of_Study",
    ["Australia", "Canada", "EU", "UK", "USA"]
)

intern_options = get_encoder_classes(
    "Internship_Experience",
    ["No", "Yes"]
)

sector_options = get_encoder_classes(
    "Job_Sector",
    ["Finance", "Healthcare", "IT", "Other"]
)


# 4. Numeric ranges from the actual dataset

def numeric_range(column, default_min, default_max, default_value):
    if raw_df is not None and column in raw_df.columns:
        s = pd.to_numeric(raw_df[column], errors="coerce").dropna()
        if len(s):
            lo = float(s.min())
            hi = float(s.max())
            val = float(s.median())
            return lo, hi, val
    return float(default_min), float(default_max), float(default_value)

age_min, age_max, age_default = numeric_range("Age", 18, 60, 24)
ysg_min, ysg_max, ysg_default = numeric_range(
    "Years_Since_Graduation", 0, 20, 2
)
gpa_min, gpa_max, gpa_default = numeric_range("GPA", 0, 4, 3.2)
salary_min, salary_max, salary_default = numeric_range(
    "Salary", 0, 200000, 50000
)

age_min, age_max = int(np.floor(age_min)), int(np.ceil(age_max))
ysg_min, ysg_max = int(np.floor(ysg_min)), int(np.ceil(ysg_max))
gpa_min, gpa_max = max(0.0, gpa_min), min(4.0, gpa_max)


# 5. Input controls

country = make_dropdown(country_options, country_options[0])
education = make_dropdown(education_options, education_options[0])
field = make_dropdown(field_options, field_options[0])
language = make_dropdown(language_options, language_options[-1])
visa = make_dropdown(visa_options, visa_options[0])
gender = make_dropdown(gender_options, gender_options[0])
ranking = make_dropdown(ranking_options, ranking_options[0])
region = make_dropdown(region_options, region_options[0])
intern = make_dropdown(
    intern_options,
    "Yes" if "Yes" in intern_options else intern_options[0]
)
sector = make_dropdown(sector_options, sector_options[0])

age = widgets.IntSlider(
    value=int(round(age_default)),
    min=age_min, max=age_max, step=1,
    layout=widgets.Layout(width="100%", height="34px"),
    style={"description_width": "0px"},
    continuous_update=False
)

years = widgets.IntSlider(
    value=int(round(ysg_default)),
    min=ysg_min, max=ysg_max, step=1,
    layout=widgets.Layout(width="100%", height="34px"),
    style={"description_width": "0px"},
    continuous_update=False
)

gpa = widgets.FloatSlider(
    value=round(gpa_default, 2),
    min=gpa_min, max=gpa_max, step=0.01,
    readout_format=".2f",
    layout=widgets.Layout(width="100%", height="34px"),
    style={"description_width": "0px"},
    continuous_update=False
)

salary = widgets.IntText(
    value=int(round(salary_default)),
    layout=widgets.Layout(width="100%", height="38px"),
    style={"description_width": "0px"}
)

def field_card(label, control, hint=""):
    return widgets.VBox([
        widgets.HTML(
            f"<div class='field-label'>{esc(label)}</div>"
            + (f"<div class='field-hint'>{esc(hint)}</div>" if hint else "")
        ),
        control
    ], layout=widgets.Layout(width="100%", margin="0 0 10px 0"))

left_col = widgets.VBox([
    field_card("Country of Origin", country),
    field_card("Education Level", education),
    field_card("Field of Study", field),
    field_card("Language Proficiency", language),
    field_card("Visa Type", visa),
    field_card("Gender", gender),
    field_card("University Ranking", ranking)
], layout=widgets.Layout(width="48%", padding="4px 12px 4px 0"))

right_col = widgets.VBox([
    field_card("Region of Study", region),
    field_card("Age", age, f"{age_min}–{age_max} years"),
    field_card("Years Since Graduation", years),
    field_card("GPA", gpa, "0–4 scale"),
    field_card("Internship Experience", intern),
    field_card("Salary", salary, "Annual salary"),
    field_card("Job Sector", sector)
], layout=widgets.Layout(width="48%", padding="4px 0 4px 12px"))


# 6. Output areas

result_output = widgets.Output()
explanation_output = widgets.Output()
history_output = widgets.Output()
history_records = []

def result_placeholder():
    return widgets.HTML("""
    <div class="result-placeholder">
        <div class="placeholder-icon">◎</div>
        <div class="placeholder-title">Ready for Prediction</div>
        <div class="placeholder-text">
            Enter the graduate profile and click <b>Predict Employability</b>.
        </div>
    </div>
    """)


# 7. Preprocessing — follows the notebook pipeline

one_hot_columns = [
    "Country_of_Origin",
    "Education_Level",
    "Field_of_Study",
    "Visa_Type",
    "Gender",
    "Region_of_Study"
]

label_columns = [
    "Language_Proficiency",
    "University_Ranking",
    "Internship_Experience",
    "Job_Sector"
]

def prepare_input():
    row = {
        "Country_of_Origin": country.value,
        "Education_Level": education.value,
        "Field_of_Study": field.value,
        "Language_Proficiency": language.value,
        "Visa_Type": visa.value,
        "Gender": gender.value,
        "University_Ranking": ranking.value,
        "Region_of_Study": region.value,
        "Age": age.value,
        "Years_Since_Graduation": years.value,
        "GPA": gpa.value,
        "Internship_Experience": intern.value,
        "Salary": salary.value,
        "Job_Sector": sector.value
    }

    data = pd.DataFrame([row])

    # Same logarithmic Salary transformation used in the notebook.
    data["Salary"] = np.log1p(
        np.maximum(pd.to_numeric(data["Salary"], errors="coerce"), 0)
    )

    # Same engineered features used in the notebook.
    data["Academic_Performance"] = (
        data["GPA"] *
        (data["Internship_Experience"].map({"Yes": 1, "No": 0}) + 1)
    )

    data["Graduate_Maturity"] = (
        data["Age"] - data["Years_Since_Graduation"]
    )

    # Same label encoders saved by the notebook.
    for col in label_columns:
        encoder = label_encoders[col]
        value = str(data.loc[0, col])
        encoder_classes = [str(x) for x in encoder.classes_]

        if value not in encoder_classes:
            if col == "Job_Sector" and "Other" in encoder_classes:
                value = "Other"
            else:
                value = encoder_classes[0]

        data[col] = encoder.transform([value])

    # Same one-hot encoding used in training.
    data = pd.get_dummies(
        data,
        columns=one_hot_columns,
        drop_first=True,
        dtype=int
    )

    # Match exact training feature order.
    data = data.reindex(columns=feature_columns, fill_value=0)

    # Reproduce the notebook's IQR clipping when df_processed is available.
    if processed_df is not None:
        numeric_cols = [
            c for c in processed_df.select_dtypes(
                include=["int64", "float64"]
            ).columns
            if c != "Employment_Status" and c in data.columns
        ]

        for col in numeric_cols:
            q1 = processed_df[col].quantile(0.25)
            q3 = processed_df[col].quantile(0.75)
            iqr = q3 - q1
            data[col] = data[col].clip(
                lower=q1 - 1.5 * iqr,
                upper=q3 + 1.5 * iqr
            )

    scaled = pd.DataFrame(
        scaler.transform(data),
        columns=feature_columns
    )

    return data, scaled


# 8. SHAP local explanation

def get_local_shap_values(scaled_input, predicted_class):
    try:
        import shap

        local_explainer = shap.TreeExplainer(best_lgbm)
        values = local_explainer.shap_values(scaled_input)

        if isinstance(values, list):
            return np.asarray(values[predicted_class]).reshape(-1)

        values = np.asarray(values)

        if values.ndim == 3:
            if values.shape[0] == 1 and values.shape[2] > 1:
                return values[0, :, predicted_class]
            if values.shape[0] > 1 and values.shape[1] == 1:
                return values[predicted_class, 0, :]
            return values[0, :, predicted_class]

        if values.ndim == 2:
            return values[0]

    except Exception:
        pass

    return None


# 9. Prediction result renderer

def render_prediction(label, probabilities, confidence):
    label_lower = str(label).lower()

    if "employ" in label_lower and "unemploy" not in label_lower:
        badge_class = "employed"
        icon = "✓"
    elif "unemploy" in label_lower:
        badge_class = "unemployed"
        icon = "!"
    else:
        badge_class = "education"
        icon = "◆"

    probability_rows = ""

    for cls, prob in sorted(
        zip(target_encoder.classes_, probabilities),
        key=lambda x: x[1],
        reverse=True
    ):
        probability_rows += f"""
        <div class="prob-row">
            <div class="prob-top">
                <span>{esc(cls)}</span>
                <b>{prob:.1%}</b>
            </div>
            <div class="prob-track">
                <div class="prob-fill" style="width:{prob*100:.1f}%"></div>
            </div>
        </div>
        """

    return widgets.HTML(f"""
    <div class="prediction-card">
        <div class="result-small-title">PREDICTION RESULT</div>
        <div class="status-badge {badge_class}">
            <span class="status-icon">{icon}</span>
            <span>{esc(label)}</span>
        </div>

        <div class="confidence-area">
            <div class="confidence-label">Prediction Confidence</div>
            <div class="confidence-value">{confidence:.1%}</div>
            <div class="confidence-track">
                <div class="confidence-fill" style="width:{confidence*100:.1f}%"></div>
            </div>
        </div>

        <div class="prob-title">Class Probabilities</div>
        {probability_rows}
    </div>
    """)

# 10. SHAP renderer

def render_explanation(scaled_input, predicted_class, predicted_label):
    shap_values_local = get_local_shap_values(
        scaled_input, predicted_class
    )

    if shap_values_local is None:
        return widgets.HTML("""
        <div class="explain-card">
            <div class="section-mini-title">MODEL EXPLANATION</div>
            <div class="muted">Local SHAP explanation could not be generated.</div>
        </div>
        """)

    values = np.asarray(shap_values_local).reshape(-1)

    if len(values) != len(feature_columns):
        return widgets.HTML("""
        <div class="explain-card">
            <div class="section-mini-title">MODEL EXPLANATION</div>
            <div class="muted">SHAP output is not compatible with the current model format.</div>
        </div>
        """)

    temp = pd.DataFrame({
        "Feature": feature_columns,
        "SHAP": values
    })
    temp["Abs"] = temp["SHAP"].abs()
    temp = temp.sort_values("Abs", ascending=False).head(8)

    rows = ""
    for _, r in temp.iterrows():
        val = float(r["SHAP"])

        if val >= 0:
            direction = "supports"
            cls = "positive"
            symbol = "+"
        else:
            direction = "opposes"
            cls = "negative"
            symbol = "−"

        rows += f"""
        <div class="shap-row">
            <div class="shap-name">{esc(r["Feature"])}</div>
            <div class="shap-direction {cls}">
                {symbol} {abs(val):.3f} · {direction} prediction
            </div>
        </div>
        """

    return widgets.HTML(f"""
    <div class="explain-card">
        <div class="section-mini-title">MODEL EXPLANATION</div>
        <div class="explain-subtitle">
            Top factors influencing the <b>{esc(predicted_label)}</b> prediction
        </div>
        {rows}
        <div class="shap-note">
            SHAP values indicate how individual features contributed to this prediction.
        </div>
    </div>
    """)


# 11. Prediction history

def render_history():
    if not history_records:
        return widgets.HTML("""
        <div class="history-empty">
            No predictions yet. Your prediction history will appear here.
        </div>
        """)

    rows = ""

    for r in reversed(history_records[-8:]):
        rows += f"""
        <tr>
            <td>{esc(r["Time"])}</td>
            <td>{esc(r["Status"])}</td>
            <td>{r["Confidence"]:.1%}</td>
            <td>{esc(r["Country"])}</td>
            <td>{esc(r["Education"])}</td>
        </tr>
        """

    return widgets.HTML(f"""
    <div class="history-table-wrap">
        <table class="history-table">
            <thead>
                <tr>
                    <th>Time</th>
                    <th>Employment Status</th>
                    <th>Confidence</th>
                    <th>Country</th>
                    <th>Education</th>
                </tr>
            </thead>
            <tbody>{rows}</tbody>
        </table>
    </div>
    """)


# 12. Main prediction function

def predict(_=None):
    with result_output:
        clear_output(wait=True)

        try:
            original_input, scaled_input = prepare_input()

            prediction = best_lgbm.predict(scaled_input)[0]
            probabilities = best_lgbm.predict_proba(scaled_input)[0]

            predicted_label = target_encoder.inverse_transform(
                [prediction]
            )[0]

            confidence = float(np.max(probabilities))

            display(render_prediction(
                predicted_label,
                probabilities,
                confidence
            ))

            with explanation_output:
                clear_output(wait=True)
                display(render_explanation(
                    scaled_input,
                    int(prediction),
                    predicted_label
                ))

            history_records.append({
                "Time": datetime.now().strftime("%H:%M:%S"),
                "Status": str(predicted_label),
                "Confidence": confidence,
                "Country": country.value,
                "Education": education.value
            })

            with history_output:
                clear_output(wait=True)
                display(render_history())

            status_message.value = (
                "<span style='color:#18794e;font-weight:700;'>"
                "✓ Prediction completed successfully"
                "</span>"
            )

        except Exception as error:
            display(widgets.HTML(f"""
            <div class="error-card">
                <b>Prediction could not be generated.</b><br>
                {esc(error)}
            </div>
            """))

            status_message.value = (
                "<span style='color:#b42318;font-weight:700;'>"
                "Prediction error — please check the input values."
                "</span>"
            )


# 13. Clear / Exit

def clear_form(_=None):
    country.value = country_options[0]
    education.value = education_options[0]
    field.value = field_options[0]
    language.value = language_options[-1]
    visa.value = visa_options[0]
    gender.value = gender_options[0]
    ranking.value = ranking_options[0]
    region.value = region_options[0]
    age.value = int(round(age_default))
    years.value = int(round(ysg_default))
    gpa.value = round(gpa_default, 2)
    intern.value = "Yes" if "Yes" in intern_options else intern_options[0]
    salary.value = int(round(salary_default))
    sector.value = sector_options[0]

    with result_output:
        clear_output(wait=True)
        display(result_placeholder())

    with explanation_output:
        clear_output(wait=True)

    status_message.value = (
        "<span style='color:#667085;'>Form cleared.</span>"
    )

def exit_prototype(_=None):
    app_container.layout.display = "none"
    print("Prototype closed. Run the prototype cell again to reopen it.")


# 14. Buttons

predict_button = widgets.Button(
    description="  PREDICT EMPLOYABILITY",
    icon="check-circle",
    button_style="primary",
    layout=widgets.Layout(width="250px", height="46px")
)

clear_button = widgets.Button(
    description="  CLEAR",
    icon="refresh",
    layout=widgets.Layout(width="120px", height="46px")
)

exit_button = widgets.Button(
    description="  EXIT",
    icon="close",
    layout=widgets.Layout(width="100px", height="46px")
)

predict_button.on_click(predict)
clear_button.on_click(clear_form)
exit_button.on_click(exit_prototype)

status_message = widgets.HTML(
    "<span style='color:#667085;'>Enter the graduate details to begin.</span>"
)

# ------------------------------------------------------------
# 15. Complete visual design
# ------------------------------------------------------------
display(HTML("""
<style>

.grand-title {
    background: linear-gradient(135deg, #12395b, #1769aa);
    color: white;
    padding: 26px 30px;
    border-radius: 14px 14px 0 0;
    text-align: center;
    box-shadow: 0 4px 14px rgba(16, 42, 67, .15);
}

.grand-title h1 {
    margin: 0;
    font-size: 28px;
    font-weight: 800;
    letter-spacing: .3px;
}

.grand-title p {
    margin: 7px 0 0;
    font-size: 14px;
    opacity: .9;
}

.section-card {
    background: #ffffff;
    border: 1px solid #dfe6ee;
    border-radius: 12px;
    padding: 20px;
    box-shadow: 0 2px 10px rgba(16, 42, 67, .06);
}

.section-heading {
    font-size: 19px;
    font-weight: 800;
    color: #173b5f;
    margin-bottom: 14px;
    padding-bottom: 10px;
    border-bottom: 2px solid #e7eef5;
}

.section-heading span {
    color: #1f7ae0;
}

.field-label {
    color: #344054;
    font-size: 13px;
    font-weight: 700;
    margin-bottom: 4px;
}

.field-hint {
    color: #98a2b3;
    font-size: 10px;
    margin-bottom: 3px;
}

.result-placeholder {
    text-align: center;
    padding: 38px 20px;
    background: #f7faff;
    border: 1px dashed #b8cee4;
    border-radius: 12px;
}

.placeholder-icon {
    font-size: 40px;
    color: #2777c9;
}

.placeholder-title {
    color: #1769aa;
    font-size: 21px;
    font-weight: 800;
    margin-top: 5px;
}

.placeholder-text {
    color: #667085;
    font-size: 13px;
    margin-top: 6px;
}

.prediction-card {
    background: #ffffff;
    border: 1px solid #dce6ef;
    border-radius: 12px;
    padding: 22px;
}

.result-small-title {
    color: #667085;
    font-size: 12px;
    font-weight: 800;
    letter-spacing: 1.2px;
    text-align: center;
}

.status-badge {
    margin: 15px auto 20px;
    width: fit-content;
    min-width: 210px;
    padding: 13px 24px;
    border-radius: 30px;
    display: flex;
    align-items: center;
    justify-content: center;
    gap: 10px;
    font-size: 20px;
    font-weight: 800;
}

.status-icon {
    font-size: 21px;
}

.status-badge.employed {
    background: #e9f8ef;
    color: #167044;
    border: 1px solid #b7e4c7;
}

.status-badge.unemployed {
    background: #fff0f0;
    color: #b42318;
    border: 1px solid #f2b8b5;
}

.status-badge.education {
    background: #eef4ff;
    color: #2457a6;
    border: 1px solid #bfd2f2;
}

.confidence-area {
    text-align: center;
    margin: 12px 0 22px;
}

.confidence-label {
    color: #475467;
    font-size: 12px;
    font-weight: 700;
}

.confidence-value {
    color: #1769aa;
    font-size: 27px;
    font-weight: 850;
    margin: 3px 0 8px;
}

.confidence-track,
.prob-track {
    background: #e9eef4;
    height: 9px;
    border-radius: 10px;
    overflow: hidden;
}

.confidence-track {
    max-width: 360px;
    margin: auto;
}

.confidence-fill,
.prob-fill {
    height: 100%;
    background: linear-gradient(90deg, #1769aa, #35a7ff);
    border-radius: 10px;
}

.prob-title {
    color: #173b5f;
    font-weight: 800;
    font-size: 14px;
    margin-bottom: 12px;
}

.prob-row {
    margin-bottom: 12px;
}

.prob-top {
    display: flex;
    justify-content: space-between;
    color: #475467;
    font-size: 12px;
    margin-bottom: 5px;
}

.explain-card {
    background: #ffffff;
    border: 1px solid #dce6ef;
    border-radius: 12px;
    padding: 20px;
}

.section-mini-title {
    color: #1769aa;
    font-size: 12px;
    font-weight: 850;
    letter-spacing: 1px;
}

.explain-subtitle {
    color: #344054;
    font-size: 13px;
    margin: 7px 0 15px;
}

.shap-row {
    padding: 10px 0;
    border-bottom: 1px solid #edf1f5;
}

.shap-name {
    color: #344054;
    font-size: 12px;
    font-weight: 700;
}

.shap-direction {
    font-size: 11px;
    margin-top: 3px;
}

.shap-direction.positive {
    color: #16824a;
}

.shap-direction.negative {
    color: #b42318;
}

.shap-note {
    margin-top: 13px;
    padding: 9px 11px;
    background: #f6f8fa;
    border-radius: 7px;
    color: #667085;
    font-size: 10px;
}

.history-table-wrap {
    overflow-x: auto;
    border: 1px solid #dce6ef;
    border-radius: 10px;
}

.history-table {
    width: 100%;
    border-collapse: collapse;
    font-size: 12px;
}

.history-table th {
    background: #173b5f;
    color: white;
    padding: 10px;
    text-align: left;
}

.history-table td {
    padding: 9px 10px;
    border-bottom: 1px solid #edf1f5;
    color: #475467;
}

.history-empty {
    padding: 20px;
    text-align: center;
    color: #98a2b3;
    background: #fafbfc;
    border: 1px dashed #d0d5dd;
    border-radius: 10px;
    font-size: 12px;
}

.error-card {
    padding: 14px;
    background: #fff1f0;
    color: #b42318;
    border: 1px solid #f3c0bd;
    border-radius: 8px;
    font-size: 12px;
}

</style>
"""))

header = widgets.HTML("""
<div class="grand-title">
    <h1>GRADUATE EMPLOYABILITY PREDICTION</h1>
    <p>Explainable Machine Learning Decision Support Prototype</p>
</div>
""")

input_heading = widgets.HTML(
    "<div class='section-heading'><span>01</span> &nbsp; Graduate Information</div>"
)

result_heading = widgets.HTML(
    "<div class='section-heading'><span>02</span> &nbsp; Prediction Result</div>"
)

explain_heading = widgets.HTML(
    "<div class='section-heading'><span>03</span> &nbsp; Explainable AI Insights</div>"
)

history_heading = widgets.HTML(
    "<div class='section-heading'><span>04</span> &nbsp; Prediction History</div>"
)

input_card = widgets.VBox(
    [input_heading, widgets.HBox([left_col, right_col])],
    layout=widgets.Layout(
        width="100%",
        padding="20px",
        border="1px solid #dfe6ee",
        border_radius="12px"
    )
)

result_card = widgets.VBox(
    [result_heading, result_output],
    layout=widgets.Layout(
        width="100%",
        padding="20px",
        border="1px solid #dfe6ee",
        border_radius="12px"
    )
)

explain_card = widgets.VBox(
    [explain_heading, explanation_output],
    layout=widgets.Layout(
        width="100%",
        padding="20px",
        border="1px solid #dfe6ee",
        border_radius="12px"
    )
)

history_card = widgets.VBox(
    [history_heading, history_output],
    layout=widgets.Layout(
        width="100%",
        padding="20px",
        border="1px solid #dfe6ee",
        border_radius="12px"
    )
)

button_bar = widgets.HBox(
    [predict_button, clear_button, exit_button],
    layout=widgets.Layout(
        justify_content="center",
        gap="12px",
        margin="15px 0 8px 0"
    )
)

footer = widgets.HTML("""
<div style="
    text-align:center;
    color:#98a2b3;
    font-size:11px;
    padding:12px 0 4px;
">
    Graduate Employability Prediction • Explainable ML Research Prototype
</div>
""")

app_container = widgets.VBox(
    [
        header,

        widgets.HTML("<div style='height:12px'></div>"),

        # 01 - Graduate Information
        input_card,

        widgets.HTML("<div style='height:8px'></div>"),

        # ACTION BUTTONS
        button_bar,

        status_message,

        widgets.HTML("<div style='height:12px'></div>"),

        # 02 - Prediction Result
        result_card,

        widgets.HTML("<div style='height:12px'></div>"),

        # 03 - Explainable AI Insights
        explain_card,

        widgets.HTML("<div style='height:12px'></div>"),

        # 04 - Prediction History
        history_card,

        footer
    ],
    layout=widgets.Layout(
        width="100%",
        padding="10px",
        overflow="visible"
    )
)
with result_output:
    display(result_placeholder())

with history_output:
    display(render_history())

display(app_container)